In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
BodyId = str
UID = str
ConnectomeReader = type(None)
print('pandas:', pd.__version__, ' polars:', pl.__version__)
import typing
import math


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# Shared minimal lookup used by the CMatrix snippets.
FIX_VNC_LOOKUP_PD = pd.DataFrame({
    "uid": ["u1", "u2", "u3"],
    "body_id": [101, 102, 103],
    "index": [0, 1, 2],
    "row_index": [0, 1, 2],
    "column_index": [0, 1, 2],
})
FIX_VNC_LOOKUP_PL = pl.from_pandas(FIX_VNC_LOOKUP_PD)

def _default_connectome_reader():
    return SimpleNamespace(name="mock-connectome-reader")

# --- vnc_cmatrix_loc_filter_item ---

# --- vnc_cmatrix_lookup_init ---
FIX_VNC_CMATRIX_LOOKUP_INIT_DEFAULT_CONNECTOME_READER = _default_connectome_reader
try:
    import scipy.sparse as _scipy_sparse
    FIX_VNC_CMATRIX_LOOKUP_INIT_SC = SimpleNamespace(sparse=_scipy_sparse)
except ImportError:
    class _CSRMatrix:
        def __init__(self, data):
            self.data = data
            self.shape = getattr(data, "shape", (len(data), len(data[0]) if len(data) else 0))
        def tocsr(self):
            return self
    FIX_VNC_CMATRIX_LOOKUP_INIT_SC = SimpleNamespace(
        sparse=SimpleNamespace(
            csr_matrix=_CSRMatrix,
            issparse=lambda x: isinstance(x, _CSRMatrix),
        )
    )

# --- vnc_cmatrix_reindex ---

# --- vnc_cmatrix_sort_tolist ---

# --- vnc_cmatrix_uid_to_index ---
FIX_VNC_CMATRIX_UID_TO_INDEX_ID_ = "u1"
FIX_VNC_CMATRIX_UID_TO_INDEX_LOOKUP = FIX_VNC_LOOKUP_PD[["uid", "row_index", "column_index"]].copy()
FIX_VNC_CMATRIX_UID_TO_INDEX_LOOKUP_PL = pl.from_pandas(FIX_VNC_CMATRIX_UID_TO_INDEX_LOOKUP)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_vnc_cmatrix_loc_filter_item():
    def __convert_index_to_uid(
        self, index: int | list[int], axis: typing.Literal["row", "column"] = "row"
    ):
        if isinstance(index, int):
            index = [index]
        lookup = self.get_lookup()
        if axis == "row":
            uids = [lookup.loc[lookup["row_index"] == id].uid.values[0] for id in index]
        elif axis == "column":
            uids = [
                lookup.loc[lookup["column_index"] == id].uid.values[0] for id in index
            ]
        else:
            raise ValueError('The axis must be either "row" or "column".')
        return uids

    def __convert_index_to_bodyid(self, index, axis="row"):
        if isinstance(index, int):
            index = [index]
        lookup = self.get_lookup()
        if axis == "row":
            body_ids = [
                lookup.loc[lookup["row_index"] == id].body_id.values[0] for id in index
            ]
        elif axis == "column":
            body_ids = [
                lookup.loc[lookup["column_index"] == id].body_id.values[0]
                for id in index
            ]
        else:
            raise ValueError('The axis must be either "row" or "column".')
        return body_ids

    def __get_uids_from_bodyids(self, body_ids: list[BodyId] | list[int]) -> list[UID]:
        lookup = self.get_lookup().copy()
        lookup = lookup[lookup["body_id"].isin(body_ids)]
        missing = set(body_ids) - set(lookup["body_id"].tolist())
        if len(missing) > 0:
            print(f"Warning: {len(missing)} body ids not found.")
        uids = lookup["uid"].tolist()
        return uids
    return __get_uids_from_bodyids

def before_vnc_cmatrix_lookup_init(default_connectome_reader, sc):
    def __init__(
        self,
        matrix: sc.sparse.csr_matrix,
        lookup: pd.DataFrame,
        CR: ConnectomeReader | None = None,
    ):
        if not sc.sparse.issparse(matrix):
            matrix = sc.sparse.csr_matrix(matrix)
        if not isinstance(matrix, sc.sparse.csr_matrix):
            matrix = matrix.tocsr()
        self.matrix = matrix

        if not isinstance(lookup, pd.DataFrame):
            raise ValueError("The lookup must be a pandas dataframe.")

        lookup = lookup.copy()
        lookup["row_index"] = lookup["index"].values
        lookup["column_index"] = lookup["index"].values
        self.lookup = lookup.drop(columns="index")

        if not all(self.lookup["row_index"].isin(range(self.matrix.shape[0]))):
            raise ValueError(
                "The lookup must include row indices up to the length of the matrix."
            )
        if not all(self.lookup["column_index"].isin(range(self.matrix.shape[1]))):
            raise ValueError(
                "The lookup must include column indices up to the length of the matrix."
            )

        self.CR = CR or default_connectome_reader()
    return __init__

def before_vnc_cmatrix_reindex():
    def __update_indexing(self):
        lookup = self.get_lookup().copy()

        lookup.sort_values(by="row_index", inplace=True)
        n_rows = lookup["row_index"].count()
        n_nan = len(lookup) - n_rows
        new_row_indices = list(range(n_rows))
        new_row_indices.extend(np.nan * np.ones(n_nan))
        lookup["row_index"] = new_row_indices

        lookup = lookup.sort_values(by="column_index")
        n_columns = lookup["column_index"].count()
        n_nan = len(lookup) - n_columns
        new_column_indices = list(range(n_columns))
        new_column_indices.extend(np.nan * np.ones(n_nan))
        lookup["column_index"] = new_column_indices

        lookup.dropna(
            axis="index",
            subset=["row_index", "column_index"],
            how="all",
            inplace=True,
        )
        if n_rows != self.matrix.shape[0]:
            raise ValueError(
                "The lookup must include row indices up to the length of the matrix."
            )
        if n_columns != self.matrix.shape[1]:
            raise ValueError(
                "The lookup must include column indices up to the length of the matrix."
            )
        self.lookup = lookup
        return
    return __update_indexing

def before_vnc_cmatrix_sort_tolist():
    def get_uids(
        self,
        sub_indices: Optional[list[int]] = None,
        axis: typing.Literal["row", "column"] = "row",
    ) -> list:
        if sub_indices is None:
            if axis == "row":
                self.lookup.sort_values(by="row_index", inplace=True)
            elif axis == "column":
                self.lookup.sort_values(by="column_index", inplace=True)
            return self.lookup["uid"].tolist()
        return self.__convert_index_to_uid(sub_indices, axis=axis)

    def get_row_indices(
        self,
        sub_uid: Optional[list] = None,
        allow_empty: bool = True,
        input_type: typing.Literal["uid", "body_id"] = "uid",
    ) -> list:
        if sub_uid is None:
            self.lookup.sort_values(by="row_index", inplace=True)
            return self.lookup["row_index"].tolist()

        if input_type == "body_id":
            sub_uid = self.__get_uids_from_bodyids(sub_uid)
        rows, _ = self.__convert_uid_to_index(sub_uid, allow_empty=allow_empty)
        if not allow_empty and len(rows) != len(sub_uid):
            raise ValueError("Some row body ids found only in the columns.")
        return rows

    def get_column_indices(
        self,
        sub_uid: Optional[list] = None,
        allow_empty: bool = True,
        input_type: typing.Literal["uid", "body_id"] = "uid",
    ) -> list:
        if sub_uid is None:
            self.lookup.sort_values(by="column_index", inplace=True)
            return self.lookup["column_index"].tolist()
        if input_type == "body_id":
            sub_uid = self.__get_uids_from_bodyids(sub_uid)
        _, columns = self.__convert_uid_to_index(sub_uid, allow_empty=allow_empty)
        if not allow_empty and len(columns) != len(sub_uid):
            raise ValueError("Some row body ids found only in the columns.")
        return columns
    return get_column_indices

def before_vnc_cmatrix_uid_to_index(id_, lookup, column_indices=None, row_indices=None):
    if column_indices is None:
        column_indices = []
    if row_indices is None:
        row_indices = []
    row_indices.append(lookup.loc[lookup["uid"] == id_].row_index.values[0])
    column_indices.append(
        lookup.loc[lookup["uid"] == id_].column_index.values[0]
    )
    row_indices = [i for i in row_indices if not pd.isna(i)]
    column_indices = [i for i in column_indices if not pd.isna(i)]
    return column_indices

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_vnc_cmatrix_loc_filter_item():
    def __convert_index_to_uid(
        self, index: int | list[int], axis: typing.Literal["row", "column"] = "row"
    ):
        if isinstance(index, int):
            index = [index]
        lookup = self.get_lookup()
        if axis == "row":
            uids = [
                lookup.filter(pl.col("row_index") == id).select("uid").to_series()[0]
                for id in index
            ]
        elif axis == "column":
            uids = [
                lookup.filter(pl.col("column_index") == id)
                .select("uid")
                .to_series()[0]
                for id in index
            ]
        else:
            raise ValueError('The axis must be either "row" or "column".')
        return uids

    def __convert_index_to_bodyid(self, index, axis="row"):
        if isinstance(index, int):
            index = [index]
        lookup = self.get_lookup()
        if axis == "row":
            body_ids = [
                lookup.filter(pl.col("row_index") == id)
                .select("body_id")
                .to_series()[0]
                for id in index
            ]
        elif axis == "column":
            body_ids = [
                lookup.filter(pl.col("column_index") == id)
                .select("body_id")
                .to_series()[0]
                for id in index
            ]
        else:
            raise ValueError('The axis must be either "row" or "column".')
        return body_ids

    def __get_uids_from_bodyids(self, body_ids: list[BodyId] | list[int]) -> list[UID]:
        lookup = self.get_lookup().clone()
        lookup = lookup.filter(pl.col("body_id").is_in(body_ids))
        missing = set(body_ids) - set(lookup.get_column("body_id").to_list())
        if len(missing) > 0:
            print(f"Warning: {len(missing)} body ids not found.")
        uids = lookup.get_column("uid").to_list()
        return uids
    return __get_uids_from_bodyids

def gen_vnc_cmatrix_lookup_init(default_connectome_reader, sc):
    def __init__(
        self,
        matrix: sc.sparse.csr_matrix,
        lookup: pl.DataFrame,
        CR: ConnectomeReader | None = None,
    ):
        if not sc.sparse.issparse(matrix):
            matrix = sc.sparse.csr_matrix(matrix)
        if not isinstance(matrix, sc.sparse.csr_matrix):
            matrix = matrix.tocsr()
        self.matrix = matrix

        if not isinstance(lookup, pl.DataFrame):
            raise ValueError("The lookup must be a pandas dataframe.")

        lookup = lookup.clone()
        lookup = lookup.with_columns(
            [
                pl.col("index").alias("row_index"),
                pl.col("index").alias("column_index"),
            ]
        )
        self.lookup = lookup.drop("index")

        if not all(self.lookup["row_index"].is_in(range(self.matrix.shape[0]))):
            raise ValueError(
                "The lookup must include row indices up to the length of the matrix."
            )
        if not all(self.lookup["column_index"].is_in(range(self.matrix.shape[1]))):
            raise ValueError(
                "The lookup must include column indices up to the length of the matrix."
            )

        self.CR = CR or default_connectome_reader()
    return __init__

def gen_vnc_cmatrix_reindex():
    def __update_indexing(self):
        lookup = self.get_lookup().clone()

        lookup = lookup.sort("row_index")
        n_rows = lookup.get_column("row_index").drop_nulls().len()
        n_nan = len(lookup) - n_rows
        new_row_indices = list(range(n_rows))
        new_row_indices.extend([None] * n_nan)
        lookup = lookup.with_columns(pl.Series("row_index", new_row_indices))

        lookup = lookup.sort("column_index")
        n_columns = lookup.get_column("column_index").drop_nulls().len()
        n_nan = len(lookup) - n_columns
        new_column_indices = list(range(n_columns))
        new_column_indices.extend([None] * n_nan)
        lookup = lookup.with_columns(pl.Series("column_index", new_column_indices))

        lookup = lookup.filter(~(pl.col("row_index").is_null() & pl.col("column_index").is_null()))
        if n_rows != self.matrix.shape[0]:
            raise ValueError(
                "The lookup must include row indices up to the length of the matrix."
            )
        if n_columns != self.matrix.shape[1]:
            raise ValueError(
                "The lookup must include column indices up to the length of the matrix."
            )
        self.lookup = lookup
        return
    return __update_indexing

def gen_vnc_cmatrix_sort_tolist():
    def get_uids(
        self,
        sub_indices: Optional[list[int]] = None,
        axis: typing.Literal["row", "column"] = "row",
    ) -> list:
        if sub_indices is None:
            if axis == "row":
                self.lookup = self.lookup.sort(by="row_index")
            elif axis == "column":
                self.lookup = self.lookup.sort(by="column_index")
            return self.lookup.get_column("uid").to_list()
        return self.__convert_index_to_uid(sub_indices, axis=axis)

    def get_row_indices(
        self,
        sub_uid: Optional[list] = None,
        allow_empty: bool = True,
        input_type: typing.Literal["uid", "body_id"] = "uid",
    ) -> list:
        if sub_uid is None:
            self.lookup = self.lookup.sort(by="row_index")
            return self.lookup.get_column("row_index").to_list()

        if input_type == "body_id":
            sub_uid = self.__get_uids_from_bodyids(sub_uid)
        rows, _ = self.__convert_uid_to_index(sub_uid, allow_empty=allow_empty)
        if not allow_empty and len(rows) != len(sub_uid):
            raise ValueError("Some row body ids found only in the columns.")
        return rows

    def get_column_indices(
        self,
        sub_uid: Optional[list] = None,
        allow_empty: bool = True,
        input_type: typing.Literal["uid", "body_id"] = "uid",
    ) -> list:
        if sub_uid is None:
            self.lookup = self.lookup.sort(by="column_index")
            return self.lookup.get_column("column_index").to_list()
        if input_type == "body_id":
            sub_uid = self.__get_uids_from_bodyids(sub_uid)
        _, columns = self.__convert_uid_to_index(sub_uid, allow_empty=allow_empty)
        if not allow_empty and len(columns) != len(sub_uid):
            raise ValueError("Some row body ids found only in the columns.")
        return columns
    return get_column_indices

def gen_vnc_cmatrix_uid_to_index(id_, lookup, column_indices=None, row_indices=None):
    if column_indices is None:
        column_indices = []
    if row_indices is None:
        row_indices = []
    row_indices.append(lookup.filter(pl.col("uid") == id_).select("row_index").to_series()[0])
    column_indices.append(
        lookup.filter(pl.col("uid") == id_).select("column_index").to_series()[0]
    )
    row_indices = [i for i in row_indices if i is not None and not (isinstance(i, float) and math.isnan(i))]
    column_indices = [i for i in column_indices if i is not None and not (isinstance(i, float) and math.isnan(i))]
    return column_indices

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: vnc_cmatrix_reindex ===

# L1 smoke – generated
try:
    _r = gen_vnc_cmatrix_reindex()
    print("✅ L1 smoke gen_vnc_cmatrix_reindex: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_vnc_cmatrix_reindex: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_vnc_cmatrix_reindex()
    print("✅ L1 smoke before_vnc_cmatrix_reindex: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_vnc_cmatrix_reindex: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – execute returned __update_indexing and compare mutated lookup.
try:
    _before_update = before_vnc_cmatrix_reindex()
    _gen_update = gen_vnc_cmatrix_reindex()
    _lookup_pd = FIX_VNC_LOOKUP_PD[["uid", "body_id", "row_index", "column_index"]].sample(frac=1, random_state=2).reset_index(drop=True)
    _lookup_pl = pl.from_pandas(_lookup_pd)
    _before_self = SimpleNamespace(matrix=np.zeros((3, 3)), lookup=_lookup_pd.copy())
    _before_self.get_lookup = lambda: _before_self.lookup
    _gen_self = SimpleNamespace(matrix=np.zeros((3, 3)), lookup=_lookup_pl)
    _gen_self.get_lookup = lambda: _gen_self.lookup
    _before_update(_before_self)
    _gen_update(_gen_self)
    compare(_before_self.lookup, _gen_self.lookup, "vnc_cmatrix_reindex lookup state", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence vnc_cmatrix_reindex: setup error — {type(_e).__name__}: {_e}")

# L3 branch – execute returned __update_indexing on unsorted lookup with null row index
try:
    _before_update = before_vnc_cmatrix_reindex()
    _gen_update = gen_vnc_cmatrix_reindex()
    _lookup_pd = pd.DataFrame({
        "uid": ["u2", "u1", "u3"],
        "body_id": [102, 101, 103],
        "row_index": [1.0, 0.0, np.nan],
        "column_index": [2, 0, 1],
    })
    _lookup_pl = pl.from_pandas(_lookup_pd)
    _before_self = SimpleNamespace(matrix=np.zeros((2, 3)), lookup=_lookup_pd.copy())
    _before_self.get_lookup = lambda: _before_self.lookup
    _gen_self = SimpleNamespace(matrix=np.zeros((2, 3)), lookup=_lookup_pl)
    _gen_self.get_lookup = lambda: _gen_self.lookup
    _before_update(_before_self)
    _gen_update(_gen_self)
    compare(_before_self.lookup, _gen_self.lookup, "L3 branch vnc_cmatrix_reindex valid", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 branch vnc_cmatrix_reindex valid: {type(_e).__name__}: {_e}")

# L3 branch – shape mismatch should raise on both implementations
try:
    _before_update = before_vnc_cmatrix_reindex()
    _gen_update = gen_vnc_cmatrix_reindex()
    _lookup_pd = pd.DataFrame({
        "uid": ["u1", "u2"],
        "body_id": [101, 102],
        "row_index": [0, 1],
        "column_index": [0, 1],
    })
    _before_self = SimpleNamespace(matrix=np.zeros((1, 2)), lookup=_lookup_pd.copy())
    _before_self.get_lookup = lambda: _before_self.lookup
    _gen_self = SimpleNamespace(matrix=np.zeros((1, 2)), lookup=pl.from_pandas(_lookup_pd))
    _gen_self.get_lookup = lambda: _gen_self.lookup
    try:
        _before_update(_before_self)
        _before_exc = None
    except Exception as _e:
        _before_exc = type(_e).__name__
    try:
        _gen_update(_gen_self)
        _gen_exc = None
    except Exception as _e:
        _gen_exc = type(_e).__name__
    if _before_exc == _gen_exc == "ValueError":
        print("✅ L3 branch vnc_cmatrix_reindex shape mismatch: MATCH")
    else:
        print(f"❌ L3 branch vnc_cmatrix_reindex shape mismatch: MISMATCH — before={_before_exc}, gen={_gen_exc}")
except Exception as _e:
    print(f"❌ L3 branch vnc_cmatrix_reindex shape mismatch setup: {type(_e).__name__}: {_e}")
